# WarehouseSort——入门 Notebook

本 Notebook 在 Easy 难度完整演示 **state IL pipeline**：安装 → 查看 environment → 下载 demos → 训练 state Diffusion Policy → 评估。

policy 读取 **privileged 低维 state vector**（proprioception、包裹位姿/标签颜色与箱子位置）。提供的 state Diffusion Policy 是起点；可选的 RGB 图像赛道更困难，见 README。

**要求：**CUDA GPU。Google Colab 中选择 *Runtime → Change runtime type → T4 GPU*。

参考：
- [ManiSkill 3](https://maniskill.readthedocs.io/en/latest/)：GPU 加速机器人仿真
- [Diffusion Policy](https://diffusion-policy.cs.columbia.edu)：Chi et al. 2023


### 已解决 episode 的样子

仅用于生成 demos 的 scripted policy 将包裹放入颜色匹配的箱子：左侧是 scene view，右侧是 policy camera。

![Easy demo](https://github.com/marso-robotics/berlin-marso-hackathon/raw/main/media/easy_demo.gif)

*(Medium 为 4 个包裹；Hard 为 6 个包裹且箱子可能互换，详见 README。)*


## 1. 安装


In [ ]:
# Install ManiSkill and dependencies (takes ~2 min on Colab)
!pip install mani-skill==3.0.1 diffusers==0.38.0 gymnasium torch torchvision hydra-core -q

# Clone the repo (skip if already in the repo directory)
import os
if not os.path.exists('warehouse_sort'):
    !git clone https://github.com/marso-robotics/berlin-marso-hackathon.git
    %cd berlin-marso-hackathon

# Install the package
!pip install -e . -q

# Colab headless rendering (offscreen Vulkan)
import os
os.environ['DISPLAY'] = ''
os.environ['PYOPENGL_PLATFORM'] = 'egl'

## 2. 查看 environment

**Easy**：2 个包裹（一个红标签、一个蓝标签）、2 个颜色箱子、位置固定。观测是 **state vector**：robot proprioception、包裹位姿/标签颜色和箱子位置/颜色。其长度依赖包裹数，因此 state policy **按难度专用**。


In [ ]:
import gymnasium as gym
import torch
import warehouse_sort  # registers WarehouseSort-v1

env = gym.make(
    'WarehouseSort-v1', num_envs=1, obs_mode='state',
    control_mode='pd_ee_delta_pos', sim_backend='gpu', render_mode='rgb_array',
    difficulty='easy', num_parcels=2, fixed_poses=True,
)

obs, _ = env.reset(seed=42)
print('state obs shape:', tuple(obs.shape))            # (1, 54) for easy (2 parcels)
print('action space   :', env.single_action_space)     # Box(-1,1,(4,))
print()
print('The state vector includes parcel poses, tag colors, and bin positions —')
print('its length grows with the parcel count, so train one model per level.')

In [ ]:
# Render the scene
from IPython.display import Image as IPImage
import PIL.Image, io

frame = env.render()                    # (1, H, W, 3) uint8
img = PIL.Image.fromarray(frame[0].cpu().numpy())
buf = io.BytesIO()
img.save(buf, format='PNG')
IPImage(buf.getvalue())

In [ ]:
# Run the scripted waypoint policy to verify the env works (it generates the demos;
# it reads privileged sim state to control the arm, so it is NOT a submittable policy).
import sys
sys.path.insert(0, '.')
from examples.scripted_policy import scripted_episode

env.close()
env = gym.make('WarehouseSort-v1', num_envs=1, obs_mode='state',
               control_mode='pd_ee_delta_pos', sim_backend='gpu',
               difficulty='easy', num_parcels=2, fixed_poses=True, max_episode_steps=200)
history = scripted_episode(env, max_steps=200, seed=42)
final_info = history[-1][-1]
print(f"Sorted: {final_info['success_count'].item():.0f} / 2")
env.close()

## 3. 示范数据（Kaggle competition data）

每个难度 200 个 episode 的 demos 是竞赛数据，主赛道为 state；无需手动录制。

- **Kaggle**：竞赛 Notebook 已自动挂载 `/kaggle/input/`，下方单元会查找数据。
- **Colab / 本地**：下方单元使用 `kagglehub` 下载。先加入竞赛，然后在 Kaggle 的 *Settings → API → Create New API Token* 创建 token，并在单元顶部填入 `username` 和 `key`。

文件会放入 `il/demos/<level>/`，因此训练命令无需变化。

> ⚠️ demos 来自 scripted policy。用它收集数据可以；提交 scripted / hard-coded controller 或读取 privileged env state 的 policy 会被取消资格。提交 policy 必须从观测做动作。


In [ ]:
# The demos (200 episodes per level) are the Kaggle COMPETITION data.
COMPETITION = 'marso-hack-berlin-2026-robot-parcel-sorting-challenge'

import os, glob, shutil, tarfile

# --- Colab / local auth (NOT needed on Kaggle) -------------------------------------------
# After joining the competition, get a token from kaggle.com -> Settings -> API ->
# 'Create New API Token' (downloads kaggle.json), then uncomment and paste its values:
# os.environ['KAGGLE_USERNAME'] = 'your_kaggle_username'
# os.environ['KAGGLE_KEY']      = 'your_kaggle_key'      # the 'key' field in kaggle.json
# (Alternatively run  `import kagglehub; kagglehub.login()`  for an interactive prompt.)
# -----------------------------------------------------------------------------------------

# 1) locate the data: a Kaggle-mounted input, else download via kagglehub
src = next((p for p in glob.glob('/kaggle/input/*')
           if glob.glob(os.path.join(p, '**/trajectory.*.pd_ee_delta_pos.physx_cuda.h5'), recursive=True)
           or glob.glob(os.path.join(p, '**/*.tar.gz'), recursive=True)), None)
if src is None:
    import kagglehub
    src = kagglehub.competition_download(COMPETITION)
print('data at:', src)

# 2) stage into il/demos/<level>/  (handles a tarball OR an easy/medium/hard folder layout)
os.makedirs('il/demos', exist_ok=True)
tars = glob.glob(os.path.join(src, '**/*.tar.gz'), recursive=True)
if tars:
    for t in tars:
        with tarfile.open(t) as tf: tf.extractall('il/demos')
else:
    for h5 in glob.glob(os.path.join(src, '**/trajectory.*.pd_ee_delta_pos.physx_cuda.h5'), recursive=True):
        lvl = os.path.basename(os.path.dirname(h5))
        os.makedirs(f'il/demos/{lvl}', exist_ok=True)
        for f in glob.glob(h5[:-2] + '*'):   # the .h5 and its .json
            shutil.copy(f, f'il/demos/{lvl}/{os.path.basename(f)}')
print('staged levels:', sorted(os.path.basename(os.path.dirname(x))
                               for x in glob.glob('il/demos/*/trajectory.*.pd_ee_delta_pos.physx_cuda.h5')))

In [ ]:
# Check the state demos are present
import glob
for f in sorted(glob.glob('il/demos/easy/*.state*.h5')):
    print(f)

### 📈 使用 TensorBoard 实时观察训练

在训练单元**之前**执行下面两行。`%tensorboard` 会启动后台 server，并嵌入会随训练写入指标自动刷新的 dashboard（loss、eval `sort_accuracy`）。


In [ ]:
%load_ext tensorboard
%tensorboard --logdir il/baselines/diffusion_policy/runs

## 4. 训练 state Diffusion Policy

[Diffusion Policy](https://diffusion-policy.cs.columbia.edu)（Chi et al. 2023）通过 action chunking 缓解普通 MLP behavior cloner 的 compounding error。

这里的短运行 `total_iters=10000` 只用于验证 pipeline；真实训练请提高至 `total_iters=30000` 以上（T4 上 Easy 约 20–40 分钟，Medium/Hard 更久）。

> ⚠️ **每个难度一个模型。**state vector 的长度随包裹数变化，Easy、Medium、Hard 必须分别训练并提交 checkpoint。


In [ ]:
# Quick training run (verify pipeline; not fully converged)
!python il/train.py method=dp demo_dir=easy \
    flags.total_iters=10000 \
    flags.eval_freq=5000 \
    flags.exp_name=warehouse_state_dp_starter

In [ ]:
# For real training (uncomment and run)
# !python il/train.py method=dp demo_dir=easy \
#     flags.total_iters=30000 \
#     flags.eval_freq=5000 \
#     flags.exp_name=warehouse_state_dp_easy

## 5. 评估 checkpoint


In [ ]:
import glob, os

# Find the latest checkpoint
ckpts = sorted(glob.glob(
    'il/baselines/diffusion_policy/runs/warehouse_state_dp_starter/checkpoints/*.pt'
))
if not ckpts:
    print('No checkpoint found — run training first')
else:
    ckpt = ckpts[-1]
    print(f'Using checkpoint: {ckpt}')
    !python eval.py difficulty=easy \
        policy=warehouse_sort.il_policy:load_dp \
        checkpoint={ckpt} \
        eval_config=conf/eval/default.yaml

**查看 rollout。** `eval.py` 会输出指标并保存 rollout 视频（render + policy-camera view）。下方显示视频；未训练模板的机械臂通常只会随机摆动，这正是训练要缩小的差距。


In [ ]:
# Display the eval rollout video (eval.py saves one under outputs/<date>/<time>/videos/)
import glob, os
from IPython.display import Video, display
vids = sorted(glob.glob('outputs/**/videos/*.mp4', recursive=True), key=os.path.getmtime)
if vids:
    print('eval rollout:', vids[-1])
    display(Video(vids[-1], embed=True, width=640))
else:
    print('No eval video found — run the eval cell above first.')

## 6. 扩展到 Medium 与 Hard

Medium（4 包裹）和 Hard（6 包裹）使用**相同 pipeline**，但 state vector 按难度变化，必须为每个难度训练独立 checkpoint。只需更改 `demo_dir` 与 `difficulty`：

```bash
python il/train.py method=dp demo_dir=medium flags.total_iters=50000 flags.exp_name=warehouse_state_dp_medium
python eval.py difficulty=medium policy=warehouse_sort.il_policy:load_dp \
    checkpoint=il/baselines/diffusion_policy/runs/warehouse_state_dp_medium/checkpoints/best_eval_sort_accuracy.pt \
    eval_config=conf/eval/default.yaml

python il/train.py method=dp demo_dir=hard flags.total_iters=60000 flags.exp_name=warehouse_state_dp_hard
python eval.py difficulty=hard policy=warehouse_sort.il_policy:load_dp \
    checkpoint=il/baselines/diffusion_policy/runs/warehouse_state_dp_hard/checkpoints/best_eval_sort_accuracy.pt \
    eval_config=conf/eval/default.yaml
```

提示：更长的 `flags.pred_horizon`（例如 32）可能有助于较长 horizon 的难度。


## 7. 可选：图像赛道（更困难）

可尝试 **RGB** 赛道：policy 只看 scene-camera image + proprioception，**没有 privileged state**。竞赛数据已包含 RGB demos；图像 shape 固定，因此一个 RGB checkpoint 可以运行在所有难度。该模板目前尚未解决任务，图像 policy 的完整分拣仍是开放问题。

```bash
python il/train.py method=dp_rgb demo_dir=easy flags.exp_name=warehouse_rgb_dp
python eval.py difficulty=easy obs_mode=rgb \
    policy=warehouse_sort.il_policy:load_dp_rgb \
    checkpoint=il/baselines/diffusion_policy/runs/warehouse_rgb_dp/checkpoints/best_eval_sort_accuracy.pt \
    eval_config=conf/eval/default.yaml
```


## 8. 下一步：如何提交

完成训练和评估后：

1. 继续训练 Medium 与 Hard，它们权重更高（0.3、0.5）。
2. 或实现其他 learned policy；满足 `act(obs, deterministic=True)` 接口即可。
3. 打包 GitHub repo、checkpoint 与声明各难度 checkpoint 的 `submission.yaml`。

完整说明见 [SUBMISSION.md](https://github.com/marso-robotics/berlin-marso-hackathon/blob/main/SUBMISSION.md)。提交内容是 codebase、`submission.yaml`、checkpoint 和能将其加载为 `act(obs)` 的 `module:function` entrypoint。

### 改进 baseline 的方向

- 增加训练量或数据：提高 `flags.total_iters`，使用 `il/gen_demos.py`。
- 调整 horizon：`flags.pred_horizon`、`flags.act_horizon`、`flags.obs_horizon`。
- 调整评估 denoising step：`load_dp` 的 `num_inference_steps`。
- 调整容量/优化：`unet_dims`、`diffusion_step_embed_dim`、`flags.batch_size`、LR。
- 提升对 held-out 宽位置与箱子互换的泛化能力。

> ⚠️ 修改训练 architecture/horizon 后，必须把同一参数传给 policy loader `load_dp(...)`，否则 checkpoint 无法加载。
